# Player Page Extraction
To get started, we need to know what URLs we need to extract the player data from.

We gotta use pandas for data manipulation and analysis as per usual. 

We also need delays between request to be within 20 requests per minute as per Basketball Reference guidelines.

In [2]:
import pandas as pd
import time

Now let's dive into data scarping specific libraries.

In [ ]:
# BeatifulSoup allows parsing and extracing data from HTML and XML documents
# We need to find specifis tables and player information within the HTML
from bs4 import BeautifulSoup

# Selenium is used for automating web browsers
from selenium import webdriver # Controls the Chrome browser
from selenium.webdriver.chrome.service import Service as ChromeService # Manges the Chrome driver installation/setup
from selenium.webdriver.chrome.options import Options # Configures Chrome settings, like headless mode, disabling images, etc.
from selenium.webdriver.common.by import By # Methods for locating elements
from selenium.webdriver.support.ui import WebDriverWait # Wait for pages to load
from selenium.webdriver.support import expected_conditions as EC # Define what to wait for 

# WebDriverManager automatically downloads and manages the Chrome driver, so we don't need to worry about it
# Converts Python code to a format that can be used by Selenium
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
print("Setting up Chrome browser...")
options = Options()
options.add_argument('--headless')  # Run in background
service = ChromeService(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

In [ ]:
# We need to configure Chrome to run efficiently for web scraping
# Headless mode means the browser runs in the background without opening a window
# We also disable images and CSS to make scraping faster

options = Options()
options.add_argument('--headless')  # Run browser in background

# Turn off security features that might block scraping
options.add_argument('--no-sandbox')  # Bypass OS security

# Use regular storage instead of in-memory storage
options.add_argument('--disable-dev-shm-usage')  # Overcome limited resource problems

# Set up the Chrome driver service
# ChromeDriverManager automatically downloads the correct driver version
service = ChromeService(ChromeDriverManager().install())

In [12]:
# Let's test our browser setup by navigating to a single Basketball Reference player index page
# We'll start with players whose last names begin with "A"
# This will confirm our browser works before scaling up to all 26 letters

# Start the Chrome browser with our configured options
driver = webdriver.Chrome(service=service, options=options)

# Navigate to the "A" players page
test_url = "https://www.basketball-reference.com/players/a/"
driver.get(test_url)

# Wait for the players table to load (Basketball Reference loads content dynamically)
# We're looking for a table with id="players"
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.ID, "players"))
)

# Check if we successfully loaded the page
page_title = driver.title
print(f"Page loaded: {page_title}")
print(f"Current URL: {driver.current_url}")

Page loaded: NBA & ABA Players with Last Names Starting with A | Basketball-Reference.com
Current URL: https://www.basketball-reference.com/players/a/


In [13]:
# Now we'll parse the HTML to extract player names and URLs
# BeautifulSoup helps us navigate the HTML structure to find specific elements

# Get the page HTML and parse it with BeautifulSoup
soup = BeautifulSoup(driver.page_source, 'html.parser')

# Find the players table (it has id="players")
players_table = soup.find('table', id='players')

# Extract player information from table rows
# We only want the first 5 players as a test
players_data = []
rows = players_table.find('tbody').find_all('tr')

print(f"Total players found on this page: {len(rows)}")

# Process first 5 players to test our extraction logic
for i, row in enumerate(rows[:5]):
    # Each player's name and URL are in a cell with data-stat="player"
    name_cell = row.find('th', {'data-stat': 'player'})
    
    if name_cell and name_cell.find('a'):
        player_link = name_cell.find('a')
        player_name = player_link.text.strip()
        # Convert relative URL to absolute URL
        player_url = "https://www.basketball-reference.com" + player_link['href']
        
        players_data.append({
            'player_name': player_name,
            'player_url': player_url
        })
        
        print(f"Player {i+1}: {player_name}")

# Convert to DataFrame to see our extracted data
df = pd.DataFrame(players_data)
print(f"\nExtracted {len(df)} players:")

# Show full URLs without truncation
pd.set_option('display.max_colwidth', None)
print(df)

# Reset display option back to default
pd.reset_option('display.max_colwidth')

Total players found on this page: 195
Player 1: Alaa Abdelnaby
Player 2: Zaid Abdul-Aziz
Player 3: Kareem Abdul-Jabbar
Player 4: Mahmoud Abdul-Rauf
Player 5: Tariq Abdul-Wahad

Extracted 5 players:
           player_name  \
0       Alaa Abdelnaby   
1      Zaid Abdul-Aziz   
2  Kareem Abdul-Jabbar   
3   Mahmoud Abdul-Rauf   
4    Tariq Abdul-Wahad   

                                                      player_url  
0  https://www.basketball-reference.com/players/a/abdelal01.html  
1  https://www.basketball-reference.com/players/a/abdulza01.html  
2  https://www.basketball-reference.com/players/a/abdulka01.html  
3  https://www.basketball-reference.com/players/a/abdulma02.html  
4  https://www.basketball-reference.com/players/a/abdulta01.html  


In [15]:
# Extract all players from the "A" page (should be around 195 players)
# This tests our extraction logic with a larger dataset

# Parse the HTML with BeautifulSoup
soup = BeautifulSoup(driver.page_source, 'html.parser')

# Find the players table
players_table = soup.find('table', id='players')

# Extract ALL player information (remove the [:5] limit)
players_data = []
rows = players_table.find('tbody').find_all('tr')

print(f"Processing all {len(rows)} players from letter A...")

# Process ALL players this time
for i, row in enumerate(rows):
    name_cell = row.find('th', {'data-stat': 'player'})
    
    if name_cell and name_cell.find('a'):
        player_link = name_cell.find('a')
        player_name = player_link.text.strip()
        player_url = "https://www.basketball-reference.com" + player_link['href']
        
        players_data.append({
            'player_name': player_name,
            'player_url': player_url
        })

# Convert to DataFrame
df_a_players = pd.DataFrame(players_data)

print(f"\nSuccessfully extracted {len(df_a_players)} players from letter A")

# Show full URLs without truncation
pd.set_option('display.max_colwidth', None)

print(f"\nFirst 5 players:")
print(df_a_players.head())
print(f"\nLast 5 players:")
print(df_a_players.tail())

# Reset display option back to default
pd.reset_option('display.max_colwidth')

# Save to CSV for backup
df_a_players.to_csv('players_letter_a.csv', index=False)
print(f"\nSaved to 'players_letter_a.csv'")

Processing all 195 players from letter A...

Successfully extracted 178 players from letter A

First 5 players:
           player_name  \
0       Alaa Abdelnaby   
1      Zaid Abdul-Aziz   
2  Kareem Abdul-Jabbar   
3   Mahmoud Abdul-Rauf   
4    Tariq Abdul-Wahad   

                                                      player_url  
0  https://www.basketball-reference.com/players/a/abdelal01.html  
1  https://www.basketball-reference.com/players/a/abdulza01.html  
2  https://www.basketball-reference.com/players/a/abdulka01.html  
3  https://www.basketball-reference.com/players/a/abdulma02.html  
4  https://www.basketball-reference.com/players/a/abdulta01.html  

Last 5 players:
          player_name  \
173      Gustavo Ayón   
174        Jeff Ayres   
175     Deandre Ayton   
176  Kelenna Azubuike   
177    Udoka Azubuike   

                                                        player_url  
173   https://www.basketball-reference.com/players/a/ayongu01.html  
174  https://www.basket

In [16]:
# Extract players from all 26 letters (A-Z)
# This will give us the complete NBA player database
# Expected total: ~5,000 players

# Start fresh browser for the full extraction
driver = webdriver.Chrome(service=service, options=options)

# Store all players from all letters
all_players = []
letters = 'abcdefghijklmnopqrstuvwxyz'

try:
    for letter in letters:
        print(f"Extracting players from letter '{letter.upper()}'...")
        
        # Navigate to the letter page
        url = f"https://www.basketball-reference.com/players/{letter}/"
        driver.get(url)
        
        # Wait for the players table to load
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "players"))
        )
        
        # Parse HTML and extract players
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        players_table = soup.find('table', id='players')
        
        if players_table:
            rows = players_table.find('tbody').find_all('tr')
            letter_count = 0
            
            for row in rows:
                name_cell = row.find('th', {'data-stat': 'player'})
                
                if name_cell and name_cell.find('a'):
                    player_link = name_cell.find('a')
                    player_name = player_link.text.strip()
                    player_url = "https://www.basketball-reference.com" + player_link['href']
                    
                    all_players.append({
                        'player_name': player_name,
                        'player_url': player_url,
                        'letter': letter.upper()
                    })
                    letter_count += 1
            
            print(f"  Found {letter_count} players")
        else:
            print(f"  No players table found for letter {letter}")
        
        # Rate limiting - 3 second delay between requests (Basketball Reference allows 20/minute)
        time.sleep(3)

finally:
    driver.quit()
    print("\nBrowser closed.")

# Create final DataFrame
df_all_players = pd.DataFrame(all_players)

print(f"\n=== EXTRACTION COMPLETE ===")
print(f"Total players extracted: {len(df_all_players)}")
print(f"Letters processed: {len(set(df_all_players['letter']))}")

# Show summary by letter
print(f"\nPlayers per letter:")
print(df_all_players['letter'].value_counts().sort_index())

# Save complete dataset
df_all_players.to_csv('all_nba_players.csv', index=False)
print(f"\nSaved complete dataset to 'all_nba_players.csv'")

Extracting players from letter 'A'...
  Found 178 players
Extracting players from letter 'B'...
  Found 508 players
Extracting players from letter 'C'...
  Found 335 players
Extracting players from letter 'D'...
  Found 264 players
Extracting players from letter 'E'...
  Found 116 players
Extracting players from letter 'F'...
  Found 161 players
Extracting players from letter 'G'...
  Found 265 players
Extracting players from letter 'H'...
  Found 376 players
Extracting players from letter 'I'...
  Found 29 players
Extracting players from letter 'J'...
  Found 263 players
Extracting players from letter 'K'...
  Found 183 players
Extracting players from letter 'L'...
  Found 209 players
Extracting players from letter 'M'...
  Found 502 players
Extracting players from letter 'N'...
  Found 112 players
Extracting players from letter 'O'...
  Found 97 players
Extracting players from letter 'P'...
  Found 238 players
Extracting players from letter 'Q'...
  Found 9 players
Extracting players